# RQ-NAC Compression on EuroSAT (Google Colab)

Train RQ-VAE models at multiple spatial sizes (**8×8**, **4×4**, **2×2**) and quantization depths (**1**, **4**, **8**) on the EuroSAT satellite dataset.

**Pipeline:**
1. Train RQ-VAE models at 9 configurations (3 spatial × 3 depths)
2. Extract codes and run N-gram Arithmetic Coding (NAC) compression
3. Compute reconstruction quality metrics (PSNR, SSIM, LPIPS, FID)
4. Train a ResNet-18 classifier on original images
5. Evaluate classification accuracy on reconstructed images at different compressions

**Setup:**
1. Go to **Runtime → Change runtime type → A100 GPU**
2. Run all cells in order
3. Each configuration trains for 150 epochs
4. Results and checkpoints are saved to Google Drive

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Clean any previous clone and re-clone fresh
!rm -rf /content/research
!git clone https://github.com/HemanthSud/OEC-Image-Processing-.git /content/research
%cd /content/research

# Install dependencies
!pip install -q omegaconf lpips PyYAML tqdm scipy

# Generate the dataset split indices file
!python split_indices.py

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
else:
    raise RuntimeError('No GPU! Go to Runtime -> Change runtime type -> T4 GPU')

# Verify split file was generated
import os
split_file = '/content/research/eurosat_split_indices.pt'
assert os.path.isfile(split_file), f'{split_file} not found! Re-run the setup cell above.'
indices = torch.load(split_file, weights_only=True)
print(f'Split indices OK: train={len(indices["train"])}, val={len(indices["val"])}, test={len(indices["test"])}')


## 2. Verify Dataset & Load Imports

In [ ]:
import os, sys, math, shutil, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from omegaconf import OmegaConf
import yaml

# Verify dataset exists
eurosat_dir = '/content/research/EuroSAT_RGB'
assert os.path.isdir(eurosat_dir), f'{eurosat_dir} not found!'
classes = sorted([c for c in os.listdir(eurosat_dir) if os.path.isdir(os.path.join(eurosat_dir, c))])
total = sum(len(os.listdir(os.path.join(eurosat_dir, c))) for c in classes)
print(f'EuroSAT: {len(classes)} classes, {total} images')

# Add rq-vae package to Python path
rqvae_root = '/content/research/rq-vae'
if rqvae_root not in sys.path:
    sys.path.insert(0, rqvae_root)

# Add nac directory to Python path
nac_root = '/content/research/nac'
if nac_root not in sys.path:
    sys.path.insert(0, nac_root)

# RQ-VAE imports
from rqvae.models.rqvae.rqvae import RQVAE
from rqvae.img_datasets.eurosat import EuroSAT, EUROSAT_CLASSES
from rqvae.img_datasets.transforms import create_transforms
from rqvae.losses.vqgan.lpips import LPIPS
from rqvae.losses.vqgan.discriminator import NLayerDiscriminator, weights_init
from rqvae.losses.vqgan.gan_loss import hinge_d_loss, vanilla_g_loss

# Metrics imports
from evaluate_metrics import compute_all_metrics, load_model_from_dir

# NAC imports
from ngram import NGramModel
from arithmetic_coding import ArithmeticEncoder

device = torch.device('cuda')
print(f'\nAll imports OK. Device: {device}')
print(f'Classes: {EUROSAT_CLASSES}')

## 3. Define Training & NAC Functions

In [ ]:
EUROSAT_DIR = '/content/research/EuroSAT_RGB'
SPLIT_INDICES = '/content/research/eurosat_split_indices.pt'

# All configs: spatial_size -> depth -> config path
ALL_CONFIGS = {}
for spatial in ['8x8', '4x4', '2x2']:
    ALL_CONFIGS[spatial] = {}
    for depth in [1, 4, 8]:
        ALL_CONFIGS[spatial][depth] = f'/content/research/rq-vae/configs/eurosat/stage1/eurosat-rqvae-{spatial}x{depth}.yaml'


def calculate_adaptive_weight(nll_loss, g_loss, last_layer):
    nll_grads = torch.autograd.grad(nll_loss, last_layer, retain_graph=True)[0]
    g_grads = torch.autograd.grad(g_loss, last_layer, retain_graph=True)[0]
    d_weight = torch.norm(nll_grads) / (torch.norm(g_grads) + 1e-4)
    return torch.clamp(d_weight, 0.0, 1e4).detach()


def train_rqvae(config_path, output_dir, epochs=150, batch_size=32, lr=4e-5):
    """Train a single RQ-VAE model from a config file. Returns model, loaders, and history."""
    os.makedirs(output_dir, exist_ok=True)

    with open(config_path) as f:
        config = OmegaConf.create(yaml.load(f, Loader=yaml.FullLoader))
    config.experiment.epochs = epochs
    config.experiment.batch_size = batch_size
    config.optimizer.init_lr = lr
    OmegaConf.save(config, os.path.join(output_dir, 'config.yaml'))

    code_shape = list(config.arch.hparams.code_shape)
    H, W, D = code_shape
    print(f'\n{"="*60}')
    print(f'TRAINING {H}x{W}x{D} | {config_path}')
    print(f'{"="*60}')

    # Datasets
    dataset_cfg = OmegaConf.create({'transforms': config.dataset.transforms})
    transforms_trn = create_transforms(dataset_cfg, split='train', is_eval=False)
    transforms_val = create_transforms(dataset_cfg, split='val', is_eval=True)
    dataset_trn = EuroSAT(EUROSAT_DIR, split='train', transform=transforms_trn,
                           split_indices_path=SPLIT_INDICES)
    dataset_val = EuroSAT(EUROSAT_DIR, split='val', transform=transforms_val,
                           split_indices_path=SPLIT_INDICES)
    loader_trn = DataLoader(dataset_trn, batch_size=batch_size, shuffle=True,
                            num_workers=2, pin_memory=True, drop_last=True)
    loader_val = DataLoader(dataset_val, batch_size=batch_size, shuffle=False,
                            num_workers=2, pin_memory=True)
    print(f'Train: {len(dataset_trn)}, Val: {len(dataset_val)}')

    # Model
    hps = dict(config.arch.hparams)
    ddconfig = dict(config.arch.ddconfig)
    model = RQVAE(**hps, ddconfig=ddconfig,
                  checkpointing=config.arch.get('checkpointing', False)).to(device)
    n_params = sum(p.numel() for p in model.parameters()) / 1e6
    print(f'Parameters: {n_params:.2f}M')

    # Discriminator
    gc = config.gan
    disc = NLayerDiscriminator(
        input_nc=gc.disc.arch.in_channels, n_layers=gc.disc.arch.num_layers,
        use_actnorm=gc.disc.arch.use_actnorm, ndf=gc.disc.arch.ndf,
    ).apply(weights_init).to(device)

    # Losses & optimizers
    lpips_loss = LPIPS().to(device).eval()
    pw = gc.loss.perceptual_weight
    dw = gc.loss.disc_weight
    gan_start = gc.loss.disc_start

    betas = tuple(config.optimizer.betas)
    opt_g = torch.optim.Adam(model.parameters(), lr=lr, betas=betas)
    opt_d = torch.optim.Adam(disc.parameters(), lr=lr, betas=betas)

    total_steps = epochs * len(loader_trn)
    warmup_steps = config.optimizer.warmup.epoch * len(loader_trn)
    def lr_lambda(step):
        if step < warmup_steps:
            return max(step / max(warmup_steps, 1), 1e-2)
        progress = (step - warmup_steps) / max(total_steps - warmup_steps, 1)
        return max(0.5 * (1 + math.cos(math.pi * progress)), 1e-2)
    sched_g = torch.optim.lr_scheduler.LambdaLR(opt_g, lr_lambda)
    sched_d = torch.optim.lr_scheduler.LambdaLR(opt_d, lr_lambda)

    # Training loop
    best_val = float('inf')
    train_hist, val_hist = [], []

    for epoch in range(epochs):
        model.train(); disc.train()
        use_gan = (epoch >= gan_start)
        epoch_recon = 0

        pbar = tqdm(loader_trn, desc=f'{H}x{W}x{D} Ep{epoch+1}/{epochs}', leave=False)
        for imgs, _ in pbar:
            imgs = imgs.to(device)
            opt_g.zero_grad()
            recon, ql, codes = model(imgs)
            lo = model.compute_loss(recon, ql, codes, xs=imgs)
            loss_recon = lo['loss_recon']
            loss_pcpt = lpips_loss(imgs, recon)

            if use_gan:
                lf, _ = disc(recon.contiguous(), None)
                loss_gen = vanilla_g_loss(lf)
                gw = calculate_adaptive_weight(loss_recon + pw * loss_pcpt,
                                               loss_gen, model.get_last_layer())
            else:
                loss_gen = torch.tensor(0., device=device)
                gw = torch.tensor(0., device=device)

            loss_g = lo['loss_total'] + pw * loss_pcpt + gw * dw * loss_gen
            loss_g.backward()
            opt_g.step(); sched_g.step()

            if use_gan:
                opt_d.zero_grad()
                lf, lr_ = disc(recon.detach(), imgs.detach())
                (dw * hinge_d_loss(lr_, lf)).backward()
                opt_d.step(); sched_d.step()

            epoch_recon += loss_recon.item()
            pbar.set_postfix(recon=f'{loss_recon.item():.4f}')

        avg_recon = epoch_recon / len(loader_trn)
        train_hist.append(avg_recon)

        # Validation
        model.eval()
        vr = 0
        with torch.no_grad():
            for imgs, _ in loader_val:
                imgs = imgs.to(device)
                r, ql, c = model(imgs)
                vr += model.compute_loss(r, ql, c, xs=imgs, valid=True)['loss_recon'].item()
        avg_val = vr / max(len(loader_val), 1)
        val_hist.append(avg_val)

        if (epoch + 1) % 10 == 0 or epoch == 0:
            print(f'  Ep{epoch+1} train_recon={avg_recon:.4f} val_recon={avg_val:.4f}')

        if avg_val < best_val:
            best_val = avg_val
            torch.save({'epoch': epoch+1, 'state_dict': model.state_dict()},
                       os.path.join(output_dir, 'best_model.pt'))

        if (epoch+1) % 25 == 0:
            torch.save({'epoch': epoch+1, 'state_dict': model.state_dict(),
                        'discriminator': disc.state_dict()},
                       os.path.join(output_dir, f'epoch{epoch+1}_model.pt'))

        if (epoch+1) % 50 == 0 or epoch == 0:
            with torch.no_grad():
                s = imgs[:8]
                sr = model(s)[0]
                grid = torch.cat([s * 0.5 + 0.5, torch.clamp(sr * 0.5 + 0.5, 0, 1)])
                torchvision.utils.save_image(
                    torchvision.utils.make_grid(grid, nrow=8),
                    os.path.join(output_dir, f'recon_ep{epoch+1:03d}.png'))

    print(f'{H}x{W}x{D} done. Best val_recon: {best_val:.4f}')
    del lpips_loss, disc
    torch.cuda.empty_cache()
    return model, config, loader_trn, loader_val, train_hist, val_hist


def extract_codes(model, config, loader_trn, loader_val):
    """Extract RQ codes from trained model and save to nac/data/."""
    code_shape = list(config.arch.hparams.code_shape)
    H, W, D = code_shape
    code_file = f'/content/research/nac/data/codes{H}x{W}x{D}.txt'
    os.makedirs(os.path.dirname(code_file), exist_ok=True)
    if os.path.exists(code_file):
        os.remove(code_file)

    model.eval()
    n_images = 0
    with torch.no_grad():
        for name, loader in [('train', loader_trn), ('val', loader_val)]:
            for imgs, _ in tqdm(loader, desc=f'Extracting {name} codes ({H}x{W}x{D})'):
                codes = model.get_codes(imgs.to(device)).cpu().numpy()
                with open(code_file, 'a') as f:
                    for i in range(codes.shape[0]):
                        f.write(' '.join(map(str, codes[i].reshape(-1).astype(int))) + '\n')
                n_images += codes.shape[0]

    print(f'Saved {n_images} sequences -> {code_file} ({H}x{W}x{D} = {H*W*D} codes/image)')
    return code_file, H, W, D


def run_nac(code_file, H, W, D, n_gram=2, k_smooth=0.1):
    """Run N-gram Arithmetic Coding on extracted codes. Returns results dict."""
    def readcode(fn, n=None):
        result = []
        with open(fn, 'r') as f:
            for i, line in enumerate(f):
                if n is not None and i >= n:
                    break
                line = line.strip()
                if line:
                    result.append([int(x) for x in line.split()])
        return result

    all_codes = readcode(code_file)
    n_train = int(len(all_codes) * 0.9)
    train_seqs = all_codes[:n_train]
    test_seqs = all_codes[n_train:]

    print(f'\nNAC {H}x{W}x{D}: {n_gram}-gram, K={k_smooth}, {len(train_seqs)} train, {len(test_seqs)} test')

    ngram_model = NGramModel(n=n_gram, k=k_smooth, start_token=-1, end_token=-2)
    ngram_model.fit(train_seqs)

    os.makedirs('/content/research/nac/models', exist_ok=True)
    ngram_model.save(f'/content/research/nac/models/{n_gram}gram_eurosat_{H}x{W}x{D}.pkl')

    encoder = ArithmeticEncoder(ngram_model=ngram_model, bits=32)
    BITS_PER_CODE = 11
    RAW_IMAGE_BITS = 64 * 64 * 3 * 8

    rates, ratios, enc_t, dec_t, errors = [], [], [], [], 0
    for seq in tqdm(test_seqs, desc=f'NAC {H}x{W}x{D}'):
        t0 = time.perf_counter()
        bits = encoder.encode(seq)
        enc_t.append((time.perf_counter() - t0) * 1000)

        t0 = time.perf_counter()
        decoded = encoder.decode(bits)
        dec_t.append((time.perf_counter() - t0) * 1000)

        if decoded != seq:
            errors += 1
        rates.append(len(bits) / (len(seq) * BITS_PER_CODE))
        ratios.append(RAW_IMAGE_BITS / len(bits))

    avg_rate = sum(rates) / len(rates)
    avg_ratio = sum(ratios) / len(ratios)
    avg_bpp = sum(len(encoder.encode(s)) for s in test_seqs) / (len(test_seqs) * 64 * 64)

    print(f'  Compression rate (vs uncompressed codes): {avg_rate:.2%}')
    print(f'  Compression ratio (vs raw RGB): {avg_ratio:.1f}x')
    print(f'  Bits per pixel (bpp): {avg_bpp:.3f}')
    print(f'  Errors: {errors}/{len(test_seqs)}')

    return {'spatial': f'{H}x{W}', 'depth': D, 'rate': avg_rate, 'ratio': avg_ratio,
            'bpp': avg_bpp, 'errors': errors, 'n_test': len(test_seqs)}


print('Functions defined.')

In [ ]:
## 4. Train All Configurations + Extract Codes + Run NAC

In [ ]:
# ─── Configuration ───
# Spatial sizes and depths to train. Adjust this list to train only what you need.
SPATIAL_SIZES = ['8x8', '4x4', '2x2']
DEPTHS = [1, 4, 8]
EPOCHS = 150
BATCH_SIZE = 32
LR = 4e-5

all_nac_results = {}
all_histories = {}
all_models_dirs = {}

for spatial in SPATIAL_SIZES:
    for depth in DEPTHS:
        key = f'{spatial}x{depth}'
        config_path = ALL_CONFIGS[spatial][depth]
        out_dir = f'/content/research/rq-vae/output/eurosat_{key}'

        model, config, loader_trn, loader_val, th, vh = train_rqvae(
            config_path, out_dir, epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LR)

        code_file, H, W, D = extract_codes(model, config, loader_trn, loader_val)
        nac_result = run_nac(code_file, H, W, D)

        all_nac_results[key] = nac_result
        all_histories[key] = {'train': th, 'val': vh}
        all_models_dirs[key] = out_dir

        del model
        torch.cuda.empty_cache()

print('\n' + '='*60)
print('ALL CONFIGURATIONS COMPLETE')
print('='*60)

In [ ]:
## 5. NAC Compression Results Table

In [ ]:
print(f'{"Config":<12} {"Codes/img":<12} {"NAC Rate":<12} {"Ratio vs RGB":<14} {"BPP":<10} {"Errors":<8}')
print('-' * 68)
for spatial in SPATIAL_SIZES:
    for depth in DEPTHS:
        key = f'{spatial}x{depth}'
        if key not in all_nac_results:
            continue
        r = all_nac_results[key]
        s = spatial.split('x')
        codes_per_img = int(s[0]) * int(s[1]) * depth
        print(f'{key:<12} {codes_per_img:<12} {r["rate"]:<12.2%} {r["ratio"]:<14.1f}x {r["bpp"]:<10.3f} {r["errors"]}/{r["n_test"]}')

## 6. Training Curves Comparison

In [ ]:
import matplotlib.pyplot as plt

n_spatial = len(SPATIAL_SIZES)
fig, axes = plt.subplots(n_spatial, 2, figsize=(14, 5 * n_spatial))
if n_spatial == 1:
    axes = axes[np.newaxis, :]

colors = {1: '#2196F3', 4: '#4CAF50', 8: '#FF9800'}

for row, spatial in enumerate(SPATIAL_SIZES):
    for depth in DEPTHS:
        key = f'{spatial}x{depth}'
        if key not in all_histories:
            continue
        label = f'D={depth}' + (' (VQ-VAE)' if depth == 1 else '')
        axes[row, 0].plot(all_histories[key]['train'], label=label, color=colors[depth])
        axes[row, 1].plot(all_histories[key]['val'], label=label, color=colors[depth])
    axes[row, 0].set_title(f'{spatial} Train Reconstruction Loss')
    axes[row, 1].set_title(f'{spatial} Val Reconstruction Loss')
    for ax in axes[row]:
        ax.set_xlabel('Epoch')
        ax.set_ylabel('MSE Loss')
        ax.legend()
        ax.grid(True, alpha=0.3)

plt.suptitle('RQ-VAE Training: Spatial Size × Depth Comparison on EuroSAT', fontsize=14)
plt.tight_layout()
plt.savefig('/content/research/rq-vae/output/training_curves_all.png', dpi=150)
plt.show()

## 7. Compression Comparison Bar Chart

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

keys = [f'{s}x{d}' for s in SPATIAL_SIZES for d in DEPTHS if f'{s}x{d}' in all_nac_results]
bpps = [all_nac_results[k]['bpp'] for k in keys]
ratios = [all_nac_results[k]['ratio'] for k in keys]
rates = [all_nac_results[k]['rate'] * 100 for k in keys]

spatial_colors = {'8x8': '#2196F3', '4x4': '#4CAF50', '2x2': '#FF9800'}
bar_colors = [spatial_colors[k.rsplit('x', 1)[0]] for k in keys]

x = np.arange(len(keys))
axes[0].bar(x, bpps, color=bar_colors)
axes[0].set_title('Bits Per Pixel (BPP)')
axes[0].set_xticks(x); axes[0].set_xticklabels(keys, rotation=45, ha='right')
for i, v in enumerate(bpps):
    axes[0].text(i, v + 0.005, f'{v:.3f}', ha='center', fontsize=8)

axes[1].bar(x, ratios, color=bar_colors)
axes[1].set_title('Compression Ratio vs Raw RGB')
axes[1].set_xticks(x); axes[1].set_xticklabels(keys, rotation=45, ha='right')
for i, v in enumerate(ratios):
    axes[1].text(i, v + 0.5, f'{v:.0f}x', ha='center', fontsize=8)

axes[2].bar(x, rates, color=bar_colors)
axes[2].set_title('NAC Rate vs Uncompressed Codes')
axes[2].set_xticks(x); axes[2].set_xticklabels(keys, rotation=45, ha='right')
for i, v in enumerate(rates):
    axes[2].text(i, v + 0.5, f'{v:.1f}%', ha='center', fontsize=8)

plt.suptitle('RQ-NAC Compression: All Configurations', fontsize=14)
plt.tight_layout()
plt.savefig('/content/research/rq-vae/output/compression_comparison_all.png', dpi=150)
plt.show()

## 8. Visualize Reconstructions (All Depths)

In [ ]:
# Load a test batch for visualization
dataset_cfg = OmegaConf.create({'transforms': {'type': 'eurosat'}})
transforms_test = create_transforms(dataset_cfg, split='val', is_eval=True)
dataset_test = EuroSAT('/content/research/EuroSAT_RGB', split='test', transform=transforms_test,
                        split_indices_path='/content/research/eurosat_split_indices.pt')
vis_loader = DataLoader(dataset_test, batch_size=8, shuffle=False, num_workers=2)
vis_imgs = next(iter(vis_loader))[0].to(device)

keys = [f'{s}x{d}' for s in SPATIAL_SIZES for d in DEPTHS
        if os.path.exists(f'/content/research/rq-vae/output/eurosat_{s}x{d}/best_model.pt')]
n_configs = len(keys)

fig, axes = plt.subplots(n_configs + 1, 8, figsize=(20, 3 * (n_configs + 1)))

# Row 0: originals
imgs_vis = (vis_imgs.cpu() * 0.5 + 0.5).clamp(0, 1)
for i in range(8):
    axes[0, i].imshow(imgs_vis[i].permute(1, 2, 0).numpy())
    axes[0, i].axis('off')
axes[0, 0].set_ylabel('Original', fontsize=11, rotation=0, labelpad=60, va='center')

# Rows 1+: reconstructions per config
for row, key in enumerate(keys, 1):
    out_dir = f'/content/research/rq-vae/output/eurosat_{key}'
    m, _ = load_model_from_dir(out_dir, device)

    with torch.no_grad():
        recon = m(vis_imgs)[0]
    recon_vis = (recon.cpu() * 0.5 + 0.5).clamp(0, 1)

    for i in range(8):
        axes[row, i].imshow(recon_vis[i].permute(1, 2, 0).numpy())
        axes[row, i].axis('off')
    axes[row, 0].set_ylabel(key, fontsize=11, rotation=0, labelpad=60, va='center')
    del m; torch.cuda.empty_cache()

plt.suptitle('Reconstruction Comparison: All Spatial Sizes × Depths', fontsize=14)
plt.tight_layout()
plt.savefig('/content/research/rq-vae/output/recon_all_comparison.png', dpi=150)
plt.show()

## 9. Quantitative Reconstruction Metrics (PSNR, SSIM, LPIPS, FID)

In [ ]:
# Compute PSNR, SSIM, LPIPS, FID on test set for all trained models
dataset_cfg = OmegaConf.create({'transforms': {'type': 'eurosat'}})
transforms_test = create_transforms(dataset_cfg, split='val', is_eval=True)
dataset_test = EuroSAT('/content/research/EuroSAT_RGB', split='test', transform=transforms_test,
                        split_indices_path='/content/research/eurosat_split_indices.pt')
test_loader = DataLoader(dataset_test, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

all_metrics = {}
for spatial in SPATIAL_SIZES:
    for depth in DEPTHS:
        key = f'{spatial}x{depth}'
        out_dir = f'/content/research/rq-vae/output/eurosat_{key}'
        if not os.path.exists(os.path.join(out_dir, 'best_model.pt')):
            print(f'Skipping {key} (no checkpoint)')
            continue

        print(f'\nComputing metrics for {key}...')
        model, config = load_model_from_dir(out_dir, device)
        metrics = compute_all_metrics(model, test_loader, device, compute_fid_flag=True)
        all_metrics[key] = metrics

        print(f'  PSNR: {metrics["psnr"]:.2f} dB | SSIM: {metrics["ssim"]:.4f} | '
              f'LPIPS: {metrics["lpips"]:.4f} | FID: {metrics["fid"]:.2f}')

        del model; torch.cuda.empty_cache()

# Print summary table
print(f'\n{"="*75}')
print(f'{"Config":<12} {"PSNR(dB)":<10} {"SSIM":<10} {"LPIPS":<10} {"FID":<10} {"BPP":<10}')
print(f'{"-"*75}')
for key in all_metrics:
    m = all_metrics[key]
    bpp = all_nac_results[key]['bpp'] if key in all_nac_results else float('nan')
    print(f'{key:<12} {m["psnr"]:<10.2f} {m["ssim"]:<10.4f} {m["lpips"]:<10.4f} {m["fid"]:<10.2f} {bpp:<10.3f}')
print(f'{"="*75}')

## 10. Train Classifier on Original EuroSAT Images

In [ ]:
def train_classifier(train_loader, val_loader, num_classes=10, epochs=30, lr=1e-3):
    """Train a ResNet-18 classifier on EuroSAT images (normalized to [0,1])."""
    from torchvision.models import resnet18, ResNet18_Weights

    classifier = resnet18(weights=ResNet18_Weights.DEFAULT)
    classifier.fc = nn.Linear(classifier.fc.in_features, num_classes)
    classifier = classifier.to(device)

    optimizer = torch.optim.Adam(classifier.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = nn.CrossEntropyLoss()

    best_acc = 0
    for epoch in range(epochs):
        # Train
        classifier.train()
        correct, total = 0, 0
        for imgs, labels in tqdm(train_loader, desc=f'Clf Ep{epoch+1}/{epochs}', leave=False):
            imgs, labels = imgs.to(device), labels.to(device)
            # Convert from [-1,1] to ImageNet normalization
            imgs_01 = imgs * 0.5 + 0.5
            mean = torch.tensor([0.485, 0.456, 0.406], device=device).view(1,3,1,1)
            std = torch.tensor([0.229, 0.224, 0.225], device=device).view(1,3,1,1)
            imgs_norm = (imgs_01 - mean) / std

            optimizer.zero_grad()
            logits = classifier(imgs_norm)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()

            correct += (logits.argmax(1) == labels).sum().item()
            total += labels.size(0)
        scheduler.step()

        # Validate
        classifier.eval()
        val_correct, val_total = 0, 0
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs, labels = imgs.to(device), labels.to(device)
                imgs_01 = imgs * 0.5 + 0.5
                imgs_norm = (imgs_01 - mean) / std
                logits = classifier(imgs_norm)
                val_correct += (logits.argmax(1) == labels).sum().item()
                val_total += labels.size(0)

        train_acc = correct / total
        val_acc = val_correct / val_total
        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f'  Ep{epoch+1} train_acc={train_acc:.4f} val_acc={val_acc:.4f}')

        if val_acc > best_acc:
            best_acc = val_acc
            torch.save(classifier.state_dict(),
                       '/content/research/rq-vae/output/classifier_best.pt')

    print(f'Best val accuracy: {best_acc:.4f}')
    classifier.load_state_dict(torch.load('/content/research/rq-vae/output/classifier_best.pt',
                                          weights_only=True))
    return classifier

# Create datasets WITH labels
dataset_cfg = OmegaConf.create({'transforms': {'type': 'eurosat'}})
transforms_trn = create_transforms(dataset_cfg, split='train', is_eval=False)
transforms_val = create_transforms(dataset_cfg, split='val', is_eval=True)

clf_train = EuroSAT('/content/research/EuroSAT_RGB', split='train', transform=transforms_trn,
                     split_indices_path='/content/research/eurosat_split_indices.pt', return_label=True)
clf_val = EuroSAT('/content/research/EuroSAT_RGB', split='val', transform=transforms_val,
                   split_indices_path='/content/research/eurosat_split_indices.pt', return_label=True)
clf_test = EuroSAT('/content/research/EuroSAT_RGB', split='test', transform=transforms_val,
                    split_indices_path='/content/research/eurosat_split_indices.pt', return_label=True)

clf_train_loader = DataLoader(clf_train, batch_size=64, shuffle=True, num_workers=2, pin_memory=True)
clf_val_loader = DataLoader(clf_val, batch_size=64, shuffle=False, num_workers=2, pin_memory=True)
clf_test_loader = DataLoader(clf_test, batch_size=64, shuffle=False, num_workers=2, pin_memory=True)

print(f'Classifier datasets: train={len(clf_train)}, val={len(clf_val)}, test={len(clf_test)}')
classifier = train_classifier(clf_train_loader, clf_val_loader)

## 11. Evaluate Classifier on Reconstructed Images

In [ ]:
@torch.no_grad()
def evaluate_classifier(classifier, test_loader, rqvae_model=None):
    """Evaluate classifier on test images (original or reconstructed through rqvae_model)."""
    classifier.eval()
    mean = torch.tensor([0.485, 0.456, 0.406], device=device).view(1,3,1,1)
    std = torch.tensor([0.229, 0.224, 0.225], device=device).view(1,3,1,1)

    correct, total = 0, 0
    for imgs, labels in test_loader:
        imgs, labels = imgs.to(device), labels.to(device)

        if rqvae_model is not None:
            rqvae_model.eval()
            imgs = rqvae_model(imgs)[0]

        imgs_01 = torch.clamp(imgs * 0.5 + 0.5, 0, 1)
        imgs_norm = (imgs_01 - mean) / std
        logits = classifier(imgs_norm)
        correct += (logits.argmax(1) == labels).sum().item()
        total += labels.size(0)

    return correct / total

# Baseline: accuracy on original test images
baseline_acc = evaluate_classifier(classifier, clf_test_loader)
print(f'Baseline accuracy (original images): {baseline_acc:.4f}')

# Evaluate on reconstructed images for each RQ-VAE config
clf_results = {'original': baseline_acc}
for spatial in SPATIAL_SIZES:
    for depth in DEPTHS:
        key = f'{spatial}x{depth}'
        out_dir = f'/content/research/rq-vae/output/eurosat_{key}'
        if not os.path.exists(os.path.join(out_dir, 'best_model.pt')):
            print(f'Skipping {key} (no checkpoint)')
            continue

        rqvae_model, _ = load_model_from_dir(out_dir, device)
        acc = evaluate_classifier(classifier, clf_test_loader, rqvae_model)
        clf_results[key] = acc
        print(f'{key}: accuracy = {acc:.4f} (delta = {acc - baseline_acc:+.4f})')
        del rqvae_model; torch.cuda.empty_cache()

# Summary table
print(f'\n{"="*50}')
print(f'{"Config":<15} {"Accuracy":<12} {"Drop":<12}')
print(f'{"-"*50}')
for key, acc in clf_results.items():
    drop = '' if key == 'original' else f'{acc - baseline_acc:+.4f}'
    print(f'{key:<15} {acc:<12.4f} {drop:<12}')
print(f'{"="*50}')

In [ ]:
# Plot classification accuracy vs compression
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Left: accuracy bar chart
keys_clf = [k for k in clf_results if k != 'original']
accs = [clf_results[k] for k in keys_clf]
spatial_colors = {'8x8': '#2196F3', '4x4': '#4CAF50', '2x2': '#FF9800'}
bar_colors = [spatial_colors.get(k.rsplit('x', 1)[0], 'gray') for k in keys_clf]

x = np.arange(len(keys_clf))
ax1.bar(x, accs, color=bar_colors)
ax1.axhline(y=baseline_acc, color='red', linestyle='--', label=f'Original ({baseline_acc:.3f})')
ax1.set_xticks(x); ax1.set_xticklabels(keys_clf, rotation=45, ha='right')
ax1.set_ylabel('Test Accuracy')
ax1.set_title('Classification Accuracy on Reconstructions')
ax1.legend()
ax1.grid(axis='y', alpha=0.3)
for i, v in enumerate(accs):
    ax1.text(i, v + 0.005, f'{v:.3f}', ha='center', fontsize=8)

# Right: accuracy vs BPP scatter
if all_nac_results:
    bpps_clr = [all_nac_results[k]['bpp'] for k in keys_clf if k in all_nac_results]
    accs_clr = [clf_results[k] for k in keys_clf if k in all_nac_results]
    labels_clr = [k for k in keys_clf if k in all_nac_results]
    colors_clr = [spatial_colors.get(k.rsplit('x', 1)[0], 'gray') for k in labels_clr]

    ax2.scatter(bpps_clr, accs_clr, c=colors_clr, s=100, edgecolors='black', zorder=5)
    for i, lbl in enumerate(labels_clr):
        ax2.annotate(lbl, (bpps_clr[i], accs_clr[i]), textcoords="offset points",
                     xytext=(5, 5), fontsize=8)
    ax2.axhline(y=baseline_acc, color='red', linestyle='--', alpha=0.5)
    ax2.set_xlabel('Bits Per Pixel (BPP)')
    ax2.set_ylabel('Classification Accuracy')
    ax2.set_title('Accuracy vs Compression (BPP)')
    ax2.grid(True, alpha=0.3)

plt.suptitle('Classification Performance Under Compression', fontsize=14)
plt.tight_layout()
plt.savefig('/content/research/rq-vae/output/classification_vs_compression.png', dpi=150)
plt.show()

## 12. Save All Results to Google Drive

In [ ]:
drive_output = '/content/drive/MyDrive/rqnac_eurosat_results'
os.makedirs(drive_output, exist_ok=True)

# Save per-config outputs
for spatial in SPATIAL_SIZES:
    for depth in DEPTHS:
        key = f'{spatial}x{depth}'
        src_dir = f'/content/research/rq-vae/output/eurosat_{key}'
        if not os.path.isdir(src_dir):
            continue
        dst_dir = os.path.join(drive_output, key)
        os.makedirs(dst_dir, exist_ok=True)
        for f in os.listdir(src_dir):
            src = os.path.join(src_dir, f)
            if os.path.isfile(src):
                shutil.copy2(src, os.path.join(dst_dir, f))

        # Copy NAC codes
        s = spatial.split('x')
        code_src = f'/content/research/nac/data/codes{s[0]}x{s[1]}x{depth}.txt'
        if os.path.exists(code_src):
            shutil.copy2(code_src, os.path.join(dst_dir, f'codes{s[0]}x{s[1]}x{depth}.txt'))

# Save comparison plots
for f in os.listdir('/content/research/rq-vae/output/'):
    if f.endswith('.png'):
        shutil.copy2(f'/content/research/rq-vae/output/{f}', os.path.join(drive_output, f))

# Save classifier
clf_src = '/content/research/rq-vae/output/classifier_best.pt'
if os.path.exists(clf_src):
    shutil.copy2(clf_src, os.path.join(drive_output, 'classifier_best.pt'))

# Save metrics and classification results as text
with open(os.path.join(drive_output, 'metrics_summary.txt'), 'w') as f:
    f.write(f'{"Config":<12} {"PSNR(dB)":<10} {"SSIM":<10} {"LPIPS":<10} {"FID":<10} {"BPP":<10} {"Clf Acc":<10}\n')
    f.write('-' * 72 + '\n')
    for key in all_metrics:
        m = all_metrics[key]
        bpp = all_nac_results.get(key, {}).get('bpp', float('nan'))
        acc = clf_results.get(key, float('nan'))
        f.write(f'{key:<12} {m["psnr"]:<10.2f} {m["ssim"]:<10.4f} {m["lpips"]:<10.4f} '
                f'{m["fid"]:<10.2f} {bpp:<10.3f} {acc:<10.4f}\n')
    f.write(f'\nBaseline classifier accuracy: {clf_results.get("original", "N/A")}\n')

print(f'All results saved to: {drive_output}')
!ls -la {drive_output}/